# ♟️ Chess Numba — Google Colab 雲端搜尋調參執行器 (Pruning Phase 3)

### 🏆 Phase 1 & 2 成果與基線鎖定
* **LMR 體系已全域鎖定**：E60 冠軍底座（`BASE=437, HIST=204, CUTNODE=1839, TTCAP=1191, MC=35, TABLE=113%`）已正式落地生產代碼。

### 🎯 Phase 3 核心前向剪枝六軸（Node-level Cutoffs & Move-loop Pruning）
1. `RFP_BASE_MULT`（當前基線 170）：反向徒勞剪枝（RFP / Static Null Move）乘數，門檻隨深度動態縮放。
2. `RAZORING_MARGIN`（當前基線 250）：剃刀剪枝邊界，靜態評估嚴重落後時提前落入靜態搜尋（QS）。
3. `FP_BASE`（當前基線 180）：徒勞剪枝（Futility Pruning）基礎邊界常數。
4. `FP_MULTIPLIER`（當前基線 125）：徒勞剪枝深度乘數（Margin = Base + Mult * d）。
5. `LMP_SCALE_PERCENT`（當前基線 107）：後期走步剪枝（Late Move Pruning）安靜步移動計數容許寬度。
6. `PRUNING_QUIET_SEE_MARGIN`（當前基線 -25）：安靜步靜態交換評估（SEE）門檻（嚴格負值保護）。

### ⚡ 調參架構
* **L1 雲端 SPSA 探索**：80 輪 × 150 局 = 12,000 局 @ 50,000 nodes（inprocess 平行加速）
* **Polyak-Ruppert 平滑取樣**：嚴格鎖定實證黃金比例 **`--tail 0.3`**（後 24 輪 / 3,600 局平滑）
* **L2 深度獨立驗收**：600 局 @ 300,000 nodes，主開局庫 `data/openings.epd`，SPRT(0.0, 3.0)

In [ ]:
# 1. 檢視 Colab 雲端硬體規格
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!python3 --version

In [ ]:
# 2. 克隆/拉取最新代碼庫並安裝相依套件
import os
branch = "feat-initial-bitboard-setup"

if not os.path.exists("/content/chess_numba"):
    %cd /content
    !git clone -b {branch} https://github.com/b08203031/chess_numba.git

%cd /content/chess_numba
!git checkout -f {branch}
!rm -f tournament_analysis/l2_verify_pruning_p3_deep_tail03.json
!git pull origin {branch}
!pip install -q numba chess numpy


In [ ]:
# 3. 測試環境與驗證 Phase 3 基準常數
import sys, json
sys.path.insert(0, "/content/chess_numba")
from tune_search.search_param_registry import get_default_params

phase3_params = [
    "RFP_BASE_MULT",
    "RAZORING_MARGIN",
    "FP_BASE",
    "FP_MULTIPLIER",
    "LMP_SCALE_PERCENT",
    "PRUNING_QUIET_SEE_MARGIN"
]

print("🔍 Phase 3 前向剪枝六軸基準值（Baseline）：")
print(json.dumps(get_default_params(phase3_params), indent=2))


In [ ]:
# 4. 啟動 L1 SPSA Phase 3 前向剪枝六軸雲端對抗調參（目標 80 輪 x 150 局 = 12,000 局）
import os
import json

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"偵測到 Colab CPU 核心數: {cpu_count}，調參並行度設為: {concurrency}")

log_file = "tune_search/logs/pruning_p3_deep_l1.jsonl"
state_file = "tune_search/states/pruning_p3_deep_l1.json"
total_campaign_iters = 80

# 讀取存檔狀態以計算剩餘需要執行的迭代輪數（支援斷點續跑）
if os.path.exists(state_file):
    with open(state_file, "r", encoding="utf-8") as f:
        saved_state = json.load(f)
    done_iters = saved_state.get("iteration", 0)
    remaining_iters = max(0, total_campaign_iters - done_iters)
    resume_flag = "--resume"
    print(f"🔄 斷點續跑模式：已完成 {done_iters}/{total_campaign_iters} 輪，本次接續執行剩餘 {remaining_iters} 輪！")
else:
    done_iters = 0
    remaining_iters = total_campaign_iters
    resume_flag = ""
    print(f"🚀 全新開跑模式：計劃執行全量 {remaining_iters} 輪！")

if remaining_iters > 0:
    !python -m tune_search.search_tuner \
      --params RFP_BASE_MULT,RAZORING_MARGIN,FP_BASE,FP_MULTIPLIER,LMP_SCALE_PERCENT,PRUNING_QUIET_SEE_MARGIN \
      --backend inprocess \
      --scale-mode integer \
      --iter {remaining_iters} \
      --games 150 \
      --nodes 50000 \
      --concurrency {concurrency} \
      --adjudicate \
      --log {log_file} \
      --state {state_file} \
      {resume_flag}
else:
    print("✅ Phase 3 的 80 輪調參已全部完成！請直接執行步驟 5 產出候選參數。\n")


In [ ]:
# 5. 分析 SPSA 調參成果，以 Polyak-Ruppert 後 30% 平均（tail 0.3）產出最佳候選參數
!python tune_search/summarize_run.py tune_search/logs/pruning_p3_deep_l1.jsonl --tail 0.3 --weight-mode uniform --out tune_search/candidate_pruning_p3_deep.json
!python tune_search/summarize_run.py tune_search/logs/pruning_p3_deep_l1.jsonl --tail 0.3 --weight-mode uniform --out tune_search/candidate_pruning_p3_tail03_uniform.json
!python tune_search/summarize_run.py tune_search/logs/pruning_p3_deep_l1.jsonl --tail 0.3 --weight-mode linear --out tune_search/candidate_pruning_p3_tail03_linear.json
print("\n=== Phase 3 等權最佳候選參數 JSON (tail 0.3 uniform) ===")
!cat tune_search/candidate_pruning_p3_deep.json
print("\n=== Phase 3 線性加權候選參數 JSON (tail 0.3 linear) ===")
!cat tune_search/candidate_pruning_p3_tail03_linear.json


In [ ]:
# 6. 啟動 L2 深度 SPRT 實戰驗證（300,000 nodes, 600 局，主開局庫獨立驗證）
# 對照基準明確指定為原始 Baseline (tune_search/baseline_pruning_p3.json)
import os

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"啟動 Phase 3 L2 深度驗收，使用並行度: {concurrency}")

!python tools/run_search_param_match.py \
  --params-json tune_search/candidate_pruning_p3_deep.json \
  --baseline-json tune_search/baseline_pruning_p3.json \
  --nodes 300000 \
  --games 600 \
  --openings data/openings.epd \
  --concurrency {concurrency} \
  --sprt 0,3 \
  --batch-games 50 \
  --adjudicate \
  --out tournament_analysis/l2_verify_pruning_p3_deep_tail03.json
